# 04 · NCCL Latency and Bandwidth：通信成本从哪里来

**本节问题：** 消息大小、GPU 数量和拓扑怎样改变 collective 成本？

完成后你应该能够：

- 建立 latency-bandwidth 直觉
- 区分 algorithm bandwidth 与 bus bandwidth
- 识别测量质量边界

前置阅读：[模块 README](../04_nccl_benchmark/README.md) · [术语表](../docs/concepts/distributed-systems-glossary.md)


## 运行状态卡

默认 `reference` 可在无 GPU 电脑上 Run All。改为 `local` 或 `gpu` 才会启动 runner。

In [ ]:
# 第一处可编辑配置：reference | local | gpu
MODE = "reference"

import sys
from pathlib import Path

notebook_dir = Path("notebooks") if Path("notebooks/_support").is_dir() else Path.cwd()
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

from _support.artifacts import load_artifact
from _support.context import create_context
from _support.plots import bar_chart
from _support.runner import run_command

ctx = create_context("04_nccl_latency_bandwidth", MODE)
ctx.card()


## 运行前预测

先写下你的预测。不要担心猜错；后面需要指出证据支持或推翻了哪一部分。


In [ ]:
PREDICTION = "我预计……，因为……"
PREDICTION

## 最小观察

这个单元只暴露关键中间状态；正式算法仍来自项目源码。

In [ ]:
alpha_us, bandwidth_gbps = 8.0, 8.0
for size in (1024, 1024**2, 64*1024**2):
    transfer_us = size * 8 / (bandwidth_gbps * 1e9) * 1e6
    print(f'{size:>9} bytes: model={alpha_us + transfer_us:.2f} µs')

## 正确性门与参考证据

读取已提交 JSON；字段缺失时立即停止，不把缺失值解释成 0。

In [ ]:
artifact_path = ctx.repo_root / '04_nccl_benchmark/results/module04_final_summary.json'
artifact = load_artifact(artifact_path, required=['schema_version', 'artifact_type', 'status', 'collective_results'])
print("证据来源：仓库参考结果", artifact_path.relative_to(ctx.repo_root))
print("顶层字段：", sorted(artifact))


In [ ]:
result = artifact['collective_results']
print('DDP payload bytes:', result['ddp_payload_bytes'])
print('large-message plateau:', result['allreduce_large_message_plateau_gbps'])
print('measurement quality:', artifact['ddp_scaling_followup']['measurement_quality'])

In [ ]:
strong = artifact['ddp_scaling_followup']['strong']
labels = ['1 GPU', '2 GPU', '4 GPU']
values = [strong[f'world_{n}']['median_samples_per_second'] for n in (1,2,4)]
bar_chart(labels, values, title='Strong scaling 中位吞吐', ylabel='samples/s')

## 本地/正式实验

命令使用参数列表在独立子进程中执行，日志和产物只写入 `_runs/`。

In [ ]:
commands = {
    "local": [sys.executable, '04_nccl_benchmark/benchmarks/check_environment.py', '--output', str(ctx.output_dir / 'environment.json')],
    "gpu": None  # 需要 NCCL_TEST_DIR；请按本章 README/四卡教程从 Terminal 启动正式 campaign,
}
command = commands.get(ctx.mode)
if ctx.mode == "reference":
    print("reference 模式：只读已提交证据，不启动实验。")
elif command is None:
    print("本节需要额外环境准备；请使用上方链接中的 Terminal 流程。")
else:
    result = run_command(
        command,
        cwd=ctx.repo_root,
        output_dir=ctx.output_dir,
        label=f"{ctx.mode}-run",
        timeout_seconds=1200,
    )
    print({"passed": result.passed, "seconds": round(result.elapsed_seconds, 2), "log": result.log_path.name})


## 与预测对照、一般规律与边界

请回答：你的预测哪一部分被支持，哪一部分被推翻？当前证据只适用于哪些硬件、shape、消息大小或软件版本？

完成实验后再阅读[正式报告](../04_nccl_benchmark/experiments/06_final_report.md)。正式报告是结论来源，Notebook 只是交互式观察层。

### 检查题

1. correctness 是否先于性能成立？
2. 当前指标的单位、重复方式和来源是什么？
3. `unavailable`、`failed` 与数值 0 为什么不能混为一谈？

下一步：[打开下一章](05_collective_algorithms.ipynb)。
